# Demo — RAG vetorial puro: o primeiro degrau

**Objetivo:** testar a busca vetorial pura com as seis perguntas do slide 14 ("O primeiro degrau: por que busca vetorial pura não basta") e ver o efeito da recuperação na resposta final. É a linha **Vetorial puro** do slide 15: uma consulta, similaridade de embedding, o menor custo.

1. **Busca vetorial pura** → a pergunta vira vetor e o índice devolve os vizinhos mais próximos;
2. **As seis perguntas do slide 14** → em que posição veio o trecho certo, e por quanto o 1º ganhou do 2º;
3. **Lupa nos empates** → ATLAS x ÓRION x HÉLIOS e SKU 48213-B x C x D;
4. **RAG completo** → a resposta com citação só é tão boa quanto o que a recuperação entregou.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** o índice `cap05-acervo`, criado e carregado pelo notebook `tipos-de-rag/00-indice-compartilhado`.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 4)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`: os **mesmos** valores usados para criar o índice no notebook 00
- `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_API_KEY`

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "azure-search-documents>=12.0.0,<13"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT") or input(
    "AZURE_SEARCH_ENDPOINT (ex.: https://meu-servico.search.windows.net): "
).strip()
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY") or getpass("AZURE_SEARCH_API_KEY (chave de administração): ")

NOME_INDICE = "cap05-acervo"
credencial_busca = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, credencial_busca)
search_client = SearchClient(AZURE_SEARCH_ENDPOINT, NOME_INDICE, credencial_busca)

# Este notebook só consulta: quem cria e carrega o índice é o notebook 00.
if NOME_INDICE not in list(index_client.list_index_names()):
    raise RuntimeError("Índice não encontrado. Rode antes o notebook tipos-de-rag/00-indice-compartilhado.")

print(f"Índice '{NOME_INDICE}' encontrado, com {search_client.get_document_count()} trechos.")

### Funções auxiliares

- `gerar_embeddings`: a mesma do notebook 00, com o mesmo deployment e a mesma dimensão. A pergunta precisa cair no mesmo espaço vetorial dos trechos.
- `responder`: a geração com citação, usada na Execução 4.
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu.

In [ ]:
pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

In [ ]:
import json

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}

# As seis perguntas do slide 14: três em que o vetor acerta e três em que ele erra.
PERGUNTAS_SLIDE_14 = ["linguagem-natural", "parafrase", "assunto-certo", "codigo-de-norma", "sigla-interna", "string-exata"]


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None

---

# Execução 1 — Busca vetorial pura

A pergunta vira vetor **no cliente**, com o mesmo `gerar_embeddings` da indexação, e o AI Search devolve os `k` trechos de vetor mais próximo (HNSW, cosseno). Nenhuma palavra é comparada: só a posição no espaço vetorial.

O `@search.score` de uma consulta vetorial vai de 0,333 a 1 (é `1 / (1 + distância de cosseno)`): quanto maior, mais próximo. Não é probabilidade de acerto.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def buscar_vetorial(pergunta: str, k: int = 5) -> list[dict]:
    """Busca vetorial pura: vetoriza a pergunta no cliente e pede ao índice os k vizinhos mais próximos."""
    vetor = gerar_embeddings([pergunta])[0]
    resultados = search_client.search(
        search_text=None,
        vector_queries=[VectorizedQuery(vector=vetor, k_nearest_neighbors=k, fields="embedding")],
        select=["id", "titulo", "secao", "conteudo", "fonte"],
        top=k,
    )
    return [
        {
            "id": r["id"],
            "titulo": r["titulo"],
            "secao": r["secao"],
            "conteudo": r["conteudo"],
            "fonte": r["fonte"],
            "score": r["@search.score"],
        }
        for r in resultados
    ]


pergunta = PERGUNTAS["linguagem-natural"]["pergunta"]
display(Markdown(f"**Pergunta:** {pergunta}"))
display(pd.DataFrame([
    {"posição": i, "score": round(t["score"], 4), "trecho": f"{t['titulo']} — {t['secao']}", "id": t["id"]}
    for i, t in enumerate(buscar_vetorial(pergunta, k=5), start=1)
]))

### O que observar

- A busca vetorial **sempre** devolve `k` resultados, mesmo quando nenhum trecho responde de verdade: no vetor puro não existe "não encontrei".
- Numa pergunta em linguagem natural, o trecho "Como pedir o reembolso de uma viagem" tende a vir no topo: pergunta e trecho falam do mesmo assunto, com palavras parecidas.

---

# Execução 2 — As seis perguntas do slide 14

Para cada pergunta, o gabarito diz qual trecho a responde. A tabela mostra **em que posição** ele apareceu no top 5 e os scores do 1º e do 2º colocados.

### O que esperar e por que pode variar

- O vetor tende a acertar as três primeiras (linguagem natural, paráfrase, assunto certo com palavras diferentes) e a tropeçar nas três últimas (código de norma, sigla interna, string exata).
- Tropeçar nem sempre é perder o 1º lugar: num acervo pequeno, o vetor pode acertar por sorte. Quando houver seções quase idênticas (NR-10 x NR-12, ATLAS x ÓRION x HÉLIOS, 48213-B x C x D), compare o score do 1º com o do 2º: **o quase empate é a lição**. A decisão está por um fio, e outra redação da pergunta pode virar o resultado.
- Os números dependem do modelo e da dimensão do embedding, e o HNSW é aproximado: não trate a tabela como gabarito fixo.

In [ ]:
resultados_slide_14 = {pid: buscar_vetorial(PERGUNTAS[pid]["pergunta"], k=5) for pid in PERGUNTAS_SLIDE_14}

linhas = []
for pid, top5 in resultados_slide_14.items():
    p = PERGUNTAS[pid]
    lugar = posicao([t["id"] for t in top5], p["trechos_esperados"][0])
    linhas.append({
        "pergunta": p["pergunta"],
        "categoria": p["categoria"],
        "o slide diz": p["o_que_o_slide_diz"],
        "posição do trecho certo": f"{lugar}º" if lugar else "fora do top 5",
        "1º colocado": f"{top5[0]['titulo']} — {top5[0]['secao']}",
        "score do 1º": round(top5[0]["score"], 4),
        "score do 2º": round(top5[1]["score"], 4),
    })

display(pd.DataFrame(linhas))

### O que observar

- Compare **posição do trecho certo** com **o slide diz**: o slide é a regra geral; a tabela é o que aconteceu neste acervo, com este modelo.
- Nas três últimas linhas, confira a distância entre **score do 1º** e **score do 2º**. Quando ela fica na segunda ou terceira casa decimal, o vetor não "sabe" a resposta: está escolhendo entre vizinhos quase iguais.
- Para o embedding, "NR-12", "ATLAS" e "48213-B" não são identificadores. São pedaços de texto parecidos com outros pedaços de texto.

---

# Execução 3 — Lupa nos empates

Abrimos o top 5 de duas perguntas do lado "o que o vetor erra": o orçamento do **projeto ATLAS** e o prazo de entrega do **SKU 48213-B**. Os três projetos têm uma seção "Orçamento e prazo" escrita com o mesmo modelo de texto, em que só mudam o nome, os números e as datas. Os três SKUs têm descrições quase idênticas, em que quase só mudam o código, o modelo do rolamento, a vedação e os números.

In [ ]:
for pid in ["sigla-interna", "string-exata"]:
    top5 = resultados_slide_14[pid]
    esperados = PERGUNTAS[pid]["trechos_esperados"][0]
    display(Markdown(f"**{PERGUNTAS[pid]['pergunta']}**"))
    display(pd.DataFrame([
        {
            "posição": i,
            "trecho": f"{t['titulo']} — {t['secao']}",
            "score": round(t["score"], 4),
            "diferença para o 1º": round(t["score"] - top5[0]["score"], 4),
            "é o trecho certo?": "sim" if t["id"] in esperados else "",
        }
        for i, t in enumerate(top5, start=1)
    ]))

### O que observar

- Os "irmãos" aparecem juntos no topo, com diferenças de score pequenas: veja a coluna **diferença para o 1º**.
- O vetor captura o **assunto** (orçamento de projeto, prazo de entrega de rolamento), não o **identificador**. "ATLAS" e "ÓRION" são nomes próprios sem significado para o embedding, e "48213-B" e "48213-C" diferem em um caractere.
- Mesmo quando o certo vem em 1º, ele ganha por pouco: outra redação da pergunta, ou outro modelo de embedding, pode colocar o vizinho na frente.

### Fala sugerida

> “Se alguém pergunta pela norma 4.557 e o sistema traz uma norma parecida, o problema não é o modelo: é que ninguém combinou vetor com palavra-chave.”

---

# Execução 4 — RAG completo

Agora o pipeline inteiro: os 3 primeiros trechos da busca vetorial vão para o modelo, que responde citando o `id` de cada trecho usado. Rodamos para uma pergunta em que o vetor tende a acertar (linguagem natural) e para uma em que ele tende a tropeçar (string exata).

### O que esperar e por que pode variar

- Se o trecho certo estiver entre os três enviados, a resposta tende a sair certa, e o mérito é da recuperação.
- Se ele ficar de fora, o modelo responde sobre o SKU errado ou diz que não há evidência suficiente. Nos dois casos, **o erro nasceu na recuperação, não no modelo** (slide 29).

In [ ]:
for pid in ["linguagem-natural", "string-exata"]:
    pergunta = PERGUNTAS[pid]["pergunta"]
    top3 = resultados_slide_14[pid][:3]
    lugar = posicao([t["id"] for t in top3], PERGUNTAS[pid]["trechos_esperados"][0])
    enviados = ", ".join(f"`{t['id']}`" for t in top3)
    situacao = f"sim, em {lugar}º lugar" if lugar else "não"

    display(Markdown(f"### {pergunta}"))
    display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
    display(Markdown(responder(pergunta, top3)))

### O que observar

- Antes de julgar a resposta, leia a linha **O trecho certo está entre eles?**: ela separa erro de recuperação de erro de geração.
- Se o trecho do 48213-B não foi enviado, nenhum prompt e nenhum modelo salvam a resposta: o modelo só sabe o que recebeu.

### Leitura arquitetural

- Roteiro de diagnóstico do slide 29: o trecho certo foi recuperado? Se não, o erro está no índice e na busca. Se foi, aí sim olhe o prompt e o modelo.
- **Remédio: busca híbrida.** O BM25 no mesmo índice acha a string exata que o vetor aproxima (slide 14). Os campos de texto com `pt-BR.lucene` já estão no índice: é o notebook 02.
- O vetorial puro continua bom para pergunta em linguagem natural e é o tipo mais barato (slide 15). O problema é usá-lo sozinho num acervo cheio de código, sigla e número: é o primeiro anti-padrão do slide 28.

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | A pergunta vira vetor e o índice sempre devolve os `k` vizinhos mais próximos | Mesmo modelo e mesma dimensão de embedding na indexação e na consulta |
| 2 | Linguagem natural, paráfrase e sinônimo tendem a acertar; código, sigla e string tendem a tropeçar | Medir com perguntas de resposta conhecida antes de escolher o tipo de RAG |
| 3 | Seções quase idênticas empatam no score | Não confiar só no vetor quando a diferença é um nome, uma letra ou um número |
| 4 | A resposta só é tão boa quanto os trechos recuperados | Diagnosticar a recuperação antes do modelo; subir para híbrido (notebook 02) |

## Frase de fechamento

> **É o RAG que quase toda empresa tem, e é sobre ele que quase toda empresa reclama: funciona bem, menos quando a pergunta é específica.**

---

# Referências oficiais

- Azure AI Search — criar uma consulta vetorial:  
  https://learn.microsoft.com/azure/search/vector-search-how-to-query

- Azure AI Search — relevância e score na busca vetorial:  
  https://learn.microsoft.com/azure/search/vector-search-ranking

- Azure AI Search — busca híbrida, o próximo degrau:  
  https://learn.microsoft.com/azure/search/hybrid-search-overview